# D growth vs fee under BTC swaps

Study whether the pool invariant `D` grows proportionally to swap fees, or more,
when swapping BTC (coin1) -> stable (coin0). Uses fork + anchor to isolate each run.

Env required:
- `WEB3_PROVIDER_URL`
- `ETHERSCAN_API_KEY`
- Optional: `YB_TBTC_POOL` (defaults to deployed tBTC pool)


In [ ]:
import os
import math
import boa

try:
    from dotenv import load_dotenv

    load_dotenv()
except Exception:
    pass


def require_env(name: str) -> str:
    v = os.environ.get(name)
    assert v, f"Missing env var: {name}"
    return v


RPC_URL = require_env("WEB3_PROVIDER_URL")
ETHERSCAN_API_KEY = require_env("ETHERSCAN_API_KEY")
POOL_ADDR = os.environ.get("YB_TBTC_POOL", "0xf1F435B05D255a5dBdE37333C0f61DA6F69c6127")
boa.fork(RPC_URL)
EOA = boa.env.generate_address()
boa.env.eoa = EOA
print("Chain:", boa.env.evm.patch.chain_id, "EOA:", EOA)

In [ ]:
# Load pool and tokens
pool = boa.from_etherscan(POOL_ADDR, api_key=ETHERSCAN_API_KEY)
stable = boa.from_etherscan(pool.coins(0), api_key=ETHERSCAN_API_KEY)
btc = boa.from_etherscan(pool.coins(1), api_key=ETHERSCAN_API_KEY)
sym0, sym1 = stable.symbol(), btc.symbol()
dec0, dec1 = int(stable.decimals()), int(btc.decimals())
print("Pool:", pool.address)
print("coin0:", stable.address, sym0, "decimals=", dec0)
print("coin1:", btc.address, sym1, "decimals=", dec1)

In [ ]:
PRECISION = 10**18


def xcp_from_D_ps(D: int, price_scale: int) -> int:
    # matches Twocrypto._xcp
    return D * PRECISION // 2 // math.isqrt(PRECISION * price_scale)


def snap():
    D = int(pool.D())
    ps = int(pool.price_scale())
    TS = int(pool.totalSupply())
    xcp = xcp_from_D_ps(D, ps)
    vp = (PRECISION * xcp) // TS if TS > 0 else 0
    return {"D": D, "price_scale": ps, "totalSupply": TS, "xcp": xcp, "virtual_price": vp}


base = snap()
base

In [ ]:
def run_swap_btc_then_measure(dx_btc: float):
    with boa.env.anchor():
        before = snap()
        dx = int(round(dx_btc * (10**dec1)))
        # fund and approve
        boa.deal(btc, EOA, dx)
        btc.approve(pool, dx, sender=EOA)

        ps0 = int(pool.price_scale())
        dy = int(pool.exchange(1, 0, dx, 0, sender=EOA))
        ps1 = int(pool.price_scale())

        after = snap()

        out = {
            "dx_btc": dx_btc,
            "dx": dx,
            "dy": dy,
            "price_scale_before": ps0,
            "price_scale_after": ps1,
            "tweaked": ps1 != ps0,
            "D_before": before["D"],
            "D_after": after["D"],
            "D_delta": after["D"] - before["D"],
            "D_delta_rel": (after["D"] - before["D"]) / before["D"],
            "xcp_before": before["xcp"],
            "xcp_after": after["xcp"],
            "xcp_delta": after["xcp"] - before["xcp"],
            "vp_before": before["virtual_price"],
            "vp_after": after["virtual_price"],
            "vp_delta": after["virtual_price"] - before["virtual_price"],
        }
        return out


for amt in [2, 4, 6, 8, 10, 20, 40, 100]:
    res = run_swap_btc_then_measure(amt)
    # Compare xcp growth (coin0 units via TS*Δvp/1e18 == Δxcp) against fee
    print(
        {
            "dx_btc": res["dx_btc"],
            "dy": res["dy"] / 1e18,
            "tweaked": res["tweaked"],
            "D_delta": res["D_delta"] / 1e18,
            "D_delta_pct": 100 * res["D_delta_rel"],
            "xcp_delta": res["xcp_delta"] / 1e18,
        }
    )